# SKAB — MLP Diagnostic Experiment

Training duration and anomaly threshold analysis using development data only.


In [ ]:
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor,as_completed
import json,time,random,hashlib,urllib.request,zipfile,platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score,recall_score,f1_score,average_precision_score,confusion_matrix
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA = ROOT / 'data' / 'SKAB'
RUN = ROOT / 'results' / 'mlp_diagnostic' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DATA.mkdir(parents=True,exist_ok=True);RUN.mkdir(parents=True,exist_ok=True)
COMMIT='b2c0d46c2971dcbfe71e26087b6d231998bb91c2'
SENSORS=['Accelerometer1RMS','Accelerometer2RMS','Current','Pressure','Temperature','Thermocouple','Voltage','Volume Flow RateRMS']
DEV_IDS=['other/1.csv', 'other/10.csv', 'other/12.csv', 'other/13.csv', 'other/14.csv', 'other/2.csv', 'other/3.csv', 'other/5.csv', 'other/6.csv', 'other/8.csv', 'other/9.csv', 'valve1/0.csv', 'valve1/10.csv', 'valve1/11.csv', 'valve1/12.csv', 'valve1/13.csv', 'valve1/14.csv', 'valve1/15.csv', 'valve1/2.csv', 'valve1/3.csv', 'valve1/4.csv', 'valve1/6.csv', 'valve1/8.csv', 'valve1/9.csv', 'valve2/0.csv', 'valve2/1.csv', 'valve2/2.csv']
FOLDS=[[['other/1.csv', 'other/12.csv', 'other/13.csv', 'other/3.csv', 'other/5.csv', 'other/6.csv', 'other/9.csv', 'valve1/10.csv', 'valve1/11.csv', 'valve1/12.csv', 'valve1/14.csv', 'valve1/15.csv', 'valve1/3.csv', 'valve1/4.csv', 'valve1/8.csv', 'valve1/9.csv', 'valve2/1.csv', 'valve2/2.csv'], ['other/10.csv', 'other/14.csv', 'other/2.csv', 'other/8.csv', 'valve1/0.csv', 'valve1/13.csv', 'valve1/2.csv', 'valve1/6.csv', 'valve2/0.csv']], [['other/10.csv', 'other/12.csv', 'other/13.csv', 'other/14.csv', 'other/2.csv', 'other/3.csv', 'other/8.csv', 'valve1/0.csv', 'valve1/10.csv', 'valve1/12.csv', 'valve1/13.csv', 'valve1/14.csv', 'valve1/2.csv', 'valve1/3.csv', 'valve1/4.csv', 'valve1/6.csv', 'valve2/0.csv', 'valve2/1.csv'], ['other/1.csv', 'other/5.csv', 'other/6.csv', 'other/9.csv', 'valve1/11.csv', 'valve1/15.csv', 'valve1/8.csv', 'valve1/9.csv', 'valve2/2.csv']], [['other/1.csv', 'other/10.csv', 'other/14.csv', 'other/2.csv', 'other/5.csv', 'other/6.csv', 'other/8.csv', 'other/9.csv', 'valve1/0.csv', 'valve1/11.csv', 'valve1/13.csv', 'valve1/15.csv', 'valve1/2.csv', 'valve1/6.csv', 'valve1/8.csv', 'valve1/9.csv', 'valve2/0.csv', 'valve2/2.csv'], ['other/12.csv', 'other/13.csv', 'other/3.csv', 'valve1/10.csv', 'valve1/12.csv', 'valve1/14.csv', 'valve1/3.csv', 'valve1/4.csv', 'valve2/1.csv']]]
WINDOW=30;SEED=0;EPOCHS=10;SNAPSHOTS=[1,10];THRESHOLDS=[.1,.5]
HIDDEN=[64,32];LR=.0003;BATCH_SIZE=512;WEIGHT_DECAY=.0001
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(2)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})
assert len(DEV_IDS)==27
assert set().union(*(set(v) for t,v in FOLDS))==set(DEV_IDS)
assert sum(len(v) for t,v in FOLDS)==27
for tr,va in FOLDS:
    assert len(tr)==18 and len(va)==9 and set(tr).isdisjoint(va)
    assert set(tr)|set(va)==set(DEV_IDS)
print('장치:',DEVICE,'| torch:',torch.__version__)
print('개발 파일27개만 사용합니다. 테스트 파일은 다운로드/평가하지 않습니다.')
print('총 3개 fold를 각각 10epoch 학습합니다.')
print('결과 폴더:',RUN)

## 1. Development Dataset


In [ ]:
files=list(DEV_IDS)
records, tables, download_manifest = [], {}, []
def download_file(fid):
    dest = DATA / fid
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        url = f'https://raw.githubusercontent.com/waico/SKAB/{COMMIT}/data/{fid}'
        last_error = None
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers={'User-Agent':'SKAB-course-project'})
                with urllib.request.urlopen(request, timeout=60) as response:
                    payload = response.read()
                tmp = dest.with_suffix('.download')
                tmp.write_bytes(payload)
                tmp.replace(dest)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
        if last_error is not None:
            raise RuntimeError(f'{fid} 다운로드 실패. 인터넷 연결 확인 후 이 셀을 다시 실행하세요.') from last_error
    return fid

with ThreadPoolExecutor(max_workers=4) as pool:
    tasks = [pool.submit(download_file, fid) for fid in files]
    for j, future in enumerate(as_completed(tasks), 1):
        future.result()
        if j % 10 == 0 or j == len(files): print(f'다운로드 준비: {j}/{len(files)}')

for j, fid in enumerate(files, 1):
    dest = DATA / fid
    df = pd.read_csv(dest, sep=';')
    required = SENSORS + ['datetime','anomaly','changepoint']
    if not set(required).issubset(df.columns):
        raise ValueError(f'{fid}의 열이 예상과 다릅니다: {df.columns.tolist()}')
    df['datetime'] = pd.to_datetime(df['datetime'], errors='raise')
    df = df.sort_values('datetime', kind='stable').reset_index(drop=True)
    if df['datetime'].duplicated().any():
        raise ValueError(f'{fid}: 중복 시각을 먼저 확인해야 합니다.')
    for col in SENSORS:
        df[col] = pd.to_numeric(df[col], errors='raise')
    if not df.anomaly.isin([0,1]).all():
        raise ValueError(f'{fid}: 정답 라벨 확인 필요')
    tables[fid] = df
    records.append({'file':fid,'family':fid.split('/')[0], 'rows':len(df),
                    'anomalies':int(df.anomaly.sum()),'anomaly_rate':df.anomaly.mean(),
                    'missing_sensors':int(df[SENSORS].isna().sum().sum()),
                    'median_interval_seconds':df.datetime.diff().dt.total_seconds().median()})
    download_manifest.append({'file':fid, 'sha256':hashlib.sha256(dest.read_bytes()).hexdigest()})
    if j % 10 == 0 or j == len(files): print(f'데이터 준비: {j}/{len(files)}')
inventory = pd.DataFrame(records)
inventory.to_csv(RUN/'data_inventory.csv',index=False,encoding='utf-8-sig')
print('총 파일:', len(inventory), '| 총 행:', inventory.rows.sum())
print('이상 비율:', f'{inventory.anomalies.sum()/inventory.rows.sum():.2%}')
print('센서 결측값:', inventory.missing_sensors.sum())
display(inventory.groupby('family').agg(files=('file','count'),rows=('rows','sum'),anomalies=('anomalies','sum')))
display(inventory[['family','median_interval_seconds']].groupby('family').describe())
display(next(iter(tables.values())).head())

## 2. Feature Engineering and Window Boundaries


In [ ]:
def features(df,mode):
    x=df[SENSORS].astype(float).replace([np.inf,-np.inf],np.nan)
    if mode=='raw':return x.copy()
    mean=x.rolling(WINDOW,min_periods=1).mean().add_suffix('_mean30')
    std=x.rolling(WINDOW,min_periods=1).std(ddof=0).add_suffix('_std30')
    delta=x.diff().add_suffix('_diff1');delta.iloc[0]=0.
    return pd.concat([x,mean,std,delta],axis=1)

def assemble(ids,mode):
    xx=[];yy=[];mm=[]
    for fid in sorted(ids):
        df=tables[fid]
        xx.append(features(df,mode));yy.append(df.anomaly.to_numpy(dtype=int))
        mm.append(pd.DataFrame({'file':fid,'row':np.arange(len(df)),'datetime':df.datetime.astype(str)}))
    return pd.concat(xx,ignore_index=True),np.concatenate(yy),pd.concat(mm,ignore_index=True)


# Rolling-window boundary example; excluded from model training.
toy=pd.DataFrame({col:[10.,12.,14.] for col in SENSORS})
toy_features=features(toy,'temporal')
col=SENSORS[0]
example=pd.DataFrame({'value':toy[col], 'past_mean':toy_features[col+'_mean30'],
                      'past_std':toy_features[col+'_std30'],'difference':toy_features[col+'_diff1']})
assert np.allclose(example.past_mean,[10,11,12])
assert np.allclose(example.difference,[0,2,2])
assert np.allclose(example.past_std,[0,1,np.sqrt(8/3)])
example.to_csv(RUN/'window_boundary_example.csv',index=False,encoding='utf-8-sig')
display(example)

## 3. MLP Training and Validation


In [ ]:
def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

class MLP(nn.Module):
    def __init__(self,n_features,hidden):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_features,hidden[0]),nn.ReLU(),
                               nn.Linear(hidden[0],hidden[1]),nn.ReLU(),nn.Linear(hidden[1],1))
    def forward(self,x):return self.net(x).squeeze(-1)


def evaluate(y,scores,threshold):
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'precision':precision_score(y,pred,zero_division=0),
            'recall':recall_score(y,pred,zero_division=0),
            'f1':f1_score(y,pred,zero_division=0),
            'pr_auc_ap':average_precision_score(y,scores),
            'false_positive_rate':float(fp/(fp+tn)),
            'predicted_anomaly_rate':float(pred.mean()),
            'score_min':float(scores.min()),'score_max':float(scores.max()),
            'score_median':float(np.median(scores)),
            'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn)}

metrics=[];learning=[];predictions=[];scaling=[]
started=time.perf_counter()
for fi,(tr,va) in enumerate(FOLDS,1):
    seed_everything(SEED)
    xtr,ytr,_=assemble(tr,'temporal');xva,yva,meta=assemble(va,'temporal')
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    train_unscaled=imputer.fit_transform(xtr).astype('float32')
    val_unscaled=imputer.transform(xva).astype('float32')
    scaler=StandardScaler().fit(train_unscaled)
    x=scaler.transform(train_unscaled).astype('float32')
    vx=scaler.transform(val_unscaled).astype('float32')
    assert x.shape[1]==32 and np.isfinite(x).all() and np.isfinite(vx).all()
    for j,col in enumerate(xtr.columns):
        scaling.append({'fold':fi,'feature':col,
          'train_mean_before':float(train_unscaled[:,j].mean()),
          'train_std_before':float(train_unscaled[:,j].std()),
          'scaler_mean':float(scaler.mean_[j]),'scaler_scale':float(scaler.scale_[j]),
          'train_mean_after':float(x[:,j].mean()),'train_std_after':float(x[:,j].std()),
          'validation_mean_after':float(vx[:,j].mean()),'validation_std_after':float(vx[:,j].std())})
    model=MLP(32,HIDDEN).to(DEVICE)
    pos_weight=torch.tensor(float((ytr==0).sum()/(ytr==1).sum()),device=DEVICE)
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    loader=DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(ytr.astype('float32'))),
                      batch_size=BATCH_SIZE,shuffle=True,generator=torch.Generator().manual_seed(SEED),num_workers=0)
    val_x=torch.from_numpy(vx).to(DEVICE);val_y=torch.from_numpy(yva.astype('float32')).to(DEVICE)
    for epoch in range(1,EPOCHS+1):
        model.train();total=0.
        for xb,yb in loader:
            xb=xb.to(DEVICE);yb=yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss=loss_fn(model(xb),yb);loss.backward();optimizer.step()
            total+=float(loss.detach().cpu())*len(yb)
        model.eval()
        with torch.no_grad():
            val_logits=model(val_x)
            val_loss=float(loss_fn(val_logits,val_y).cpu())
            scores=torch.sigmoid(val_logits).cpu().numpy()
        learning.append({'fold':fi,'epoch':epoch,'train_bce':total/len(ytr),'validation_bce':val_loss})
        if epoch in SNAPSHOTS:
            for threshold in THRESHOLDS:
                metrics.append({'fold':fi,'epoch':epoch,'threshold':threshold,**evaluate(yva,scores,threshold)})
            pp=meta.copy();pp['fold']=fi;pp['epoch']=epoch;pp['y_true']=yva;pp['score']=scores
            pp['prediction_t01']=(scores>=.1).astype(int);pp['prediction_t05']=(scores>=.5).astype(int)
            predictions.append(pp)
            print(f'Fold {fi}/3 | epoch {epoch:02d}/10 | 학습 BCE {total/len(ytr):.4f} | 검증 BCE {val_loss:.4f}',flush=True)
    pd.DataFrame(metrics).to_csv(RUN/'validation_metrics.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(learning).to_csv(RUN/'learning_curves.csv',index=False,encoding='utf-8-sig')
    del model,optimizer,val_x,val_y
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    print(f'Fold {fi}/3 완료 | 누적 {(time.perf_counter()-started)/60:.1f}분',flush=True)

detail=pd.DataFrame(metrics);losses=pd.DataFrame(learning)
scaling=pd.DataFrame(scaling)
scaling.to_csv(RUN/'standardization_audit.csv',index=False,encoding='utf-8-sig')
predictions=pd.concat(predictions,ignore_index=True)
predictions.to_csv(RUN/'validation_predictions.csv.gz',index=False,compression='gzip')
cols=['precision','recall','f1','pr_auc_ap','false_positive_rate','predicted_anomaly_rate','score_min','score_max','score_median']
summary=detail.groupby(['epoch','threshold'])[cols].mean().reset_index()
summary.to_csv(RUN/'validation_summary.csv',index=False,encoding='utf-8-sig')
display(summary.round(4))
print('표는 3개 검증 fold의 평균입니다. 테스트 성능이나 새 최종 모델 선택 결과가 아닙니다.')

## 4. Visualization and Export Results


In [ ]:
mean_loss=losses.groupby('epoch')[['train_bce','validation_bce']].mean()
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(mean_loss.index,mean_loss.train_bce,label='Train',color='#1E3A5F')
ax.plot(mean_loss.index,mean_loss.validation_bce,label='Validation',color='#4A90D9')
ax.set_xlabel('Epoch');ax.set_ylabel('Weighted BCE, mean of 3 folds');ax.legend();fig.tight_layout()
fig.savefig(RUN/'learning_comparison.png',dpi=190);plt.show()

fig,axs=plt.subplots(1,2,figsize=(10,3.8),sharex=True)
for ax,epoch in zip(axs,SNAPSHOTS):
    data=predictions[predictions.epoch==epoch]
    for label,color,name in [(0,'#1E3A5F','Normal'),(1,'#4A90D9','Anomaly')]:
        ax.hist(data.loc[data.y_true==label,'score'],bins=np.linspace(0,1,31),density=True,histtype='step',linewidth=1.7,color=color,label=name)
    ax.axvline(.1,color='gray',linestyle='--',label='Threshold 0.1')
    ax.axvline(.5,color='black',linestyle=':',label='Threshold 0.5')
    ax.set_title(f'Epoch {epoch}');ax.set_xlabel('Anomaly score');ax.set_ylabel('Density')
axs[0].legend(fontsize=8);fig.tight_layout();fig.savefig(RUN/'score_distributions.png',dpi=190);plt.show()

notes={'purpose':'Reviewer Q2 diagnosis: epoch 1 vs 10; thresholds 0.1 vs 0.5',
 'data':'Original 27 development files only; no test file used',
 'development_ids':DEV_IDS,'folds':[{'train':t,'validation':v} for t,v in FOLDS],
 'fixed':{'hidden':HIDDEN,'lr':LR,'weight_decay':WEIGHT_DECAY,'batch_size':BATCH_SIZE,
          'seed':SEED,'window':WINDOW,'loss':'BCEWithLogitsLoss, train-derived pos_weight',
          'scaling':'32 features, StandardScaler fitted on fold-training data only'},
 'changed':{'epochs':SNAPSHOTS,'thresholds':THRESHOLDS},
 'early_stopping':False,'point_adjustment':False,
 'interpretation':['Diagnostic study, not replacement final-performance evaluation',
 'Fixed 10 epochs tests the hypothesis; not assumed to improve performance',
 'One training seed with 3 fixed validation folds; not proof of a single causal mechanism'],
 'data_sha256':download_manifest,'versions':{'python':platform.python_version(),'torch':str(torch.__version__),
 'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__,'device':str(DEVICE)}}
(RUN/'diagnostic_manifest.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=RUN.parent/(RUN.name+'_review_mlp_diagnostic.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('Diagnostic experiment complete.')
print('Results archive:', zip_path)
try:
    from google.colab import files as colab_files
except ImportError:
    pass
else:
    colab_files.download(str(zip_path))